# Image Analysis Based Fusion Rate Calculation Experiment Analysis
This notebook is intended to be used for plotting the fusion counts over a 6 day period to analyze the fusion rate of HCC1806 and MDA-MB-231 cells

- Use environment.yml

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import re

In [ ]:
def plot_variable_vs_time(dataframe, sample_ID_df, xlabel=None, ylabel=None, title=None, grid_lines=True, plot_separately=False, save_svg=False, fig_size=(10, 6), font_sz=18, point_markers=False):

    # --- Publication style settings ---
    mpl.rcParams.update({
        'font.family': 'Arial',
        'font.weight': 'bold',
        'axes.labelweight': 'bold',
        'axes.titleweight': 'bold',
        'font.size': font_sz,
        'axes.titlesize': font_sz,
        'axes.labelsize': font_sz,
        'xtick.labelsize': font_sz - 2,
        'ytick.labelsize': font_sz - 2,
        'legend.fontsize': font_sz - 8,
        'axes.linewidth': 1.5,
        'xtick.major.width': 1.5,
        'ytick.major.width': 1.5,
        'xtick.major.size': 5,
        'ytick.major.size': 5,
        'lines.linewidth': 2,
        'errorbar.capsize': 4,
    })

    dataframe = dataframe.apply(pd.to_numeric, errors='coerce')
    time_values = dataframe.loc[:, 'Elapsed']

    has_error_bars = any('SE' in column for column in dataframe.columns)

    # Build yerr_df up front so it can be used for y-axis scaling too
    yerr_df = pd.DataFrame()
    if has_error_bars:
        for column in dataframe.columns:
            for sample in sample_ID_df.columns:
                if 'SE' in column and sample_ID_df.at['key', sample] in column:
                    yerr_df[sample] = dataframe[column].values

    # Compute max_y based on mean + SE so error bars aren't clipped by the y-axis limit.
    if has_error_bars:
        upper_bounds = []
        for column in dataframe.columns:
            for sample in sample_ID_df.columns:
                if sample_ID_df.at['key', sample] in column and 'SE' not in column:
                    upper_bounds.append(dataframe[column] + yerr_df[sample])
        max_y = pd.concat(upper_bounds, axis=1).max().max() if upper_bounds else dataframe.drop(['Elapsed', 'Date Time'], axis=1, errors='ignore').max().max()
    else:
        max_y = dataframe.drop(['Elapsed', 'Date Time'], axis=1, errors='ignore').max().max()

    y_padding = (max_y) * 0.1

    marker = 'o' if point_markers else ''

    def style_axes(ax):
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(1.5)
        ax.spines['bottom'].set_linewidth(1.5)
        ax.yaxis.set_ticks_position('left')
        ax.xaxis.set_ticks_position('bottom')
        if grid_lines:
            ax.grid(True, linestyle='--', linewidth=0.6, alpha=0.5, color='gray')
        ax.set_axisbelow(True)

    def save_and_show(fig, plot_title, save_svg):
        if save_svg:
            filename = f"{plot_title}.svg" if plot_title else "plot.svg"
            fig.savefig(filename, format='svg', bbox_inches='tight', dpi=300)
        plt.show()

    # --- Combined plot ---
    fig, ax = plt.subplots(figsize=fig_size)
    ax.set_ylim(0, max_y + y_padding)
    style_axes(ax)

    if has_error_bars:
        for column in dataframe.columns:
            for sample in sample_ID_df.columns:
                if sample_ID_df.at['key', sample] in column and 'SE' not in column:
                    ax.errorbar(time_values, dataframe[column], yerr=yerr_df[sample],
                                color=sample_ID_df.at['color', sample], label=column,
                                linewidth=2, capsize=4, capthick=1.5, elinewidth=1.5,
                                marker=marker)
    else:
        for column in dataframe.columns:
            for sample in sample_ID_df.columns:
                if sample_ID_df.at['key', sample] in column:
                    ax.plot(time_values, dataframe[column], label=column,
                            color=sample_ID_df.at['color', sample], linewidth=2,
                            marker=marker)

    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)

    if xlabel:
        ax.set_xlabel(xlabel, labelpad=10)
    if ylabel:
        ax.set_ylabel(ylabel, labelpad=10)
    if title:
        ax.set_title(title, pad=12, fontweight='bold')

    fig.tight_layout()
    save_and_show(fig, title, save_svg)

    # --- Separate plots ---
    if plot_separately:
        if has_error_bars:
            for column in dataframe.columns:
                for sample in sample_ID_df.columns:
                    if sample_ID_df.at['key', sample] in column and 'SE' not in column:
                        fig, ax = plt.subplots(figsize=fig_size)
                        ax.errorbar(time_values, dataframe[column], yerr=yerr_df[sample],
                                    color=sample_ID_df.at['color', sample], label=column,
                                    linewidth=2, capsize=4, capthick=1.5, elinewidth=1.5,
                                    marker=marker)
                        ax.set_ylim(0, max_y + y_padding)
                        style_axes(ax)
                        ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)
                        if xlabel: ax.set_xlabel(xlabel, labelpad=10)
                        if ylabel: ax.set_ylabel(ylabel, labelpad=10)
                        plot_title = title + f' - {column}' if title else f'Plot for {column}'
                        ax.set_title(plot_title, pad=12, fontweight='bold')
                        fig.tight_layout()
                        save_and_show(fig, plot_title, save_svg)
        else:
            for column in dataframe.columns:
                for sample in sample_ID_df.columns:
                    if sample_ID_df.at['key', sample] in column:
                        fig, ax = plt.subplots(figsize=fig_size)
                        ax.plot(time_values, dataframe[column], label=column,
                                color=sample_ID_df.at['color', sample], linewidth=2,
                                marker=marker)
                        ax.set_ylim(0, max_y + y_padding)
                        style_axes(ax)
                        ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)
                        if xlabel: ax.set_xlabel(xlabel, labelpad=10)
                        if ylabel: ax.set_ylabel(ylabel, labelpad=10)
                        plot_title = title + f' - {column}' if title else f'Plot for {column}'
                        ax.set_title(plot_title, pad=12, fontweight='bold')
                        fig.tight_layout()
                        save_and_show(fig, plot_title, save_svg)

In [ ]:
def reshape_incucyte_data(df):
    df = df.copy()
    df.columns = df.columns.str.strip()

    # Rename to short, consistent names
    col_map = {
        'Incucyte Image Time': 'Time',
        'Plate Well and Image #': 'Well',
        '# Red Cells': 'Red',
        '# Green Cells': 'Green',
        'New Fusion Events': 'Fusion',
        'Total (Green+Red)': 'Total',
        'Normalized Fusion Rate/Day': 'Rate'
    }
    df = df.rename(columns=col_map)

    # Extract the day number (e.g. "3d00hr00m" -> 3) as the Elapsed integer
    df['Elapsed'] = df['Time'].str.extract(r'^(\d+)d').astype(int)

    # Pivot: one row per Time, one column per (measurement, well) pair
    wide = df.pivot_table(
        index='Time',
        columns='Well',
        values=['Red', 'Green', 'Fusion', 'Total', 'Rate'],
        aggfunc='first'   # change/remove if there are multiple images per well/time to combine
    )

    # Flatten MultiIndex columns -> "Red_B2", "Green_B2", etc.
    wide.columns = [f'{measure}_{well}' for measure, well in wide.columns]
    wide = wide.reset_index().rename(columns={'Time': 'Date Time'})

    # Fill missing values with 0 for count-based measures (Red, Green, Fusion, Total).
    # Rate is left as NaN since a missing normalized rate isn't meaningfully "zero".
    fill_measures = ('Red', 'Green', 'Fusion', 'Total')
    fill_cols = [c for c in wide.columns if c.split('_', 1)[0] in fill_measures]
    wide[fill_cols] = wide[fill_cols].fillna(0)

    # Re-attach Elapsed (one value per unique Time)
    elapsed_map = df.drop_duplicates('Time').set_index('Time')['Elapsed']
    wide.insert(1, 'Elapsed', wide['Date Time'].map(elapsed_map))

    # Order columns: Date Time, Elapsed, then grouped by well (Red, Green, Fusion, Total, Rate)
    measure_order = ['Red', 'Green', 'Fusion', 'Total', 'Rate']
    other_cols = [c for c in wide.columns if c not in ('Date Time', 'Elapsed')]
    def sort_key(c):
        measure, well = c.rsplit('_', 1)
        return (well, measure_order.index(measure))
    other_cols = sorted(other_cols, key=sort_key)

    wide = wide[['Date Time', 'Elapsed'] + other_cols]
    wide = wide.sort_values('Elapsed').reset_index(drop=True)

    return wide

In [ ]:
def calculate_sample_average_with_se(dataframe, sample_ID_df):
    """
    Calculate sample averages and standard errors for each sample based on the given DataFrame and sample ID DataFrame.
    
    Parameters:
    dataframe (pd.DataFrame): The input DataFrame containing raw data.
    sample_ID_df (pd.DataFrame): The DataFrame containing sample IDs and related information.
    
    Returns:
    pd.DataFrame: A DataFrame containing sample averages and standard errors.
    """
    dataframe = dataframe.apply(pd.to_numeric, errors='coerce')
    dataframe.columns = dataframe.columns.astype(str)

    # Isolate columns containing each sample's keyword
    columns_dict = {}
    for sample in sample_ID_df.columns:
        columns_dict[sample] = [col for col in dataframe.columns if sample_ID_df.at['key', sample] in col]

    result_df = pd.DataFrame({'Elapsed': dataframe.loc[:, 'Elapsed']})

    # Calculate the average and standard error for selected columns at each timepoint
    for sample in sample_ID_df.columns:
        average_values = dataframe[columns_dict[sample]].mean(axis=1)
        avg_column_name = 'Average_' + sample_ID_df.at['key',sample]
        result_df[avg_column_name] = average_values
        se_values = dataframe[columns_dict[sample]].sem(axis=1)
        se_column_name = 'SE_' + sample_ID_df.at['key',sample]
        result_df[se_column_name] = se_values

    return result_df


## 231s

### kennedy

In [ ]:
file_path = "/stor/work/Brock/kennedy/SC_repo/data/ImageBasedFusionRateAnalysis/MDA-MB-231_Fusion_Rate.csv"
df = pd.read_csv(file_path)
raw_231_data = reshape_incucyte_data(df)

sample_ID_df = pd.read_csv('/stor/work/Brock/kennedy/SC_repo/data/ImageBasedFusionRateAnalysis/fusionRate_sample_ID.csv',index_col=0)
average_231_data = calculate_sample_average_with_se(raw_231_data,sample_ID_df)

plot_variable_vs_time(average_231_data.loc[average_231_data['Elapsed']<7,['Elapsed','Average_Fusion', 'SE_Fusion']], sample_ID_df, point_markers=True, save_svg=True, grid_lines=False, xlabel='Days', ylabel='New Fusion Events Observed (# Events)', title='MDA-MB-231 Count vs. Time')

### Average Fusion Rate
- Average of (new fusion events)/(total cells in well) across all 6 days

In [ ]:
average_rate_231 = average_231_data['Average_Rate'].iloc[:6].mean()
print(f'Average Fusion Event Rate over 6 Days: {average_rate_231}')
events_permil_231 = average_rate_231*1000000
print(f'\nAverage new fusion events per 1 M cells: {events_permil_231}')

## 1806s

In [ ]:
file_path = "/stor/work/Brock/kennedy/SC_repo/data/ImageBasedFusionRateAnalysis/HCC1806_FusionRate.csv"
df = pd.read_csv(file_path)
raw_1806_data = reshape_incucyte_data(df)

sample_ID_df = pd.read_csv('/stor/work/Brock/kennedy/SC_repo/data/ImageBasedFusionRateAnalysis/fusionRate_sample_ID.csv',index_col=0)
average_1806_data = calculate_sample_average_with_se(raw_1806_data,sample_ID_df)

plot_variable_vs_time(average_1806_data.loc[average_1806_data['Elapsed'],['Elapsed','Average_Fusion', 'SE_Fusion']], sample_ID_df, point_markers=True, save_svg=True, grid_lines=False, xlabel='Days', ylabel='New Fusion Events Observed (# Events)', title='HCC1806 Count vs. Time')

### Average Fusion Rate
- Average of (new fusion events)/(total cells in well) across all 6 days

In [ ]:
average_rate_1806 = average_1806_data['Average_Rate'].iloc[:6].mean()
print(f'Average Fusion Event Rate over 6 Days: {average_rate_1806}')
events_permil_1806 = average_rate_1806*1000000
print(f'\nAverage new fusion events per 1 M cells: {events_permil_1806}')